# Phase 2 — Data Quality

Raw Data
   ↓
1. Missing values
   ↓
2. Duplicates
   ↓
3. Invalid values
   ↓
4. Wrong data types
   ↓
5. Inconsistent values
   ↓
6. Decide treatment
   ↓
7. Create cleaned datasets
   ↓
8. Validate cleaned datasets
   ↓
9. Commit to GitHub

 Our cleaned data will eventually go into: data/processed/

 If we make a mistake while cleaning, we can always go back to the original.

In [2]:
import pandas as pd

In [3]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

products = pd.read_csv("../data/raw/olist_products_dataset.csv")

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

In [4]:
def missing_summary(df):
    missing=df.isna().sum()
    percentage=(missing/len(df)*100).round(2)

    return pd.DataFrame({
        "missing_count":missing,
        "missing_percentage":percentage
    }).sort_values(
        "missing_percentage",
        ascending=False
    )

## For orders

In [5]:
missing_summary(orders)

,missing_count,missing_percentage
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16
order_id,0,0.00
order_purchase_timestamp,0,0.00
order_status,0,0.00
customer_id,0,0.00
order_estimated_delivery_date,0,0.00


In [6]:
#This tells us: Among orders with no customer-delivery date, what is their status


# ex: no delivery date and cancelled=619
#OR ex: 619 cancelled orders have no delivery date

# ex: no delivery date and shipped 1107
#OR ex: 1,107 shipped orders have no delivery date
orders[
    orders["order_delivered_customer_date"].isna()
]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [7]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
orders[
    orders["order_approved_at"].isna()
]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [9]:
orders[
    orders["order_delivered_carrier_date"].isna()
]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [10]:

#The first checks duplicate order IDs.
orders["order_id"].duplicated().sum()

np.int64(0)

In [11]:
#The second checks whether the entire row is duplicated.
orders.duplicated().sum()

np.int64(0)

In [12]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [13]:
#Why sort it?
#Because we want the columns with the most missing values at the top.
order_items.isna().sum().sort_values(ascending=False)

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [14]:
(order_items["price"]<0).sum()

np.int64(0)

In [15]:
(order_items["freight_value"]<0).sum()

np.int64(0)

In [16]:
# "Is the value missing?"
(order_items["price"].isnull()).sum()

#"Is the value zero?"
(order_items["price"]==0).sum()


np.int64(0)

Check product dimensions

In [17]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [18]:
(products["product_weight_g"]<0).sum()

np.int64(0)

In [19]:
#Single bracket → one column
#Double bracket → multiple columns
#The inner [] is a Python list containing the column names:
#The outer [] tells Pandas:"Select these columns from the DataFrame."

(products[["product_height_cm","product_length_cm","product_width_cm"]]<0).sum()

product_height_cm    0
product_length_cm    0
product_width_cm     0
dtype: int64

Check reviews


In [20]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

The review score should be within the expected rating range.

In [21]:
reviews["review_score"].min()

np.int64(1)

In [22]:
reviews["review_score"].max()

np.int64(5)

Step 3: Fix data types

In [23]:
#Identify the date columns

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

errors="raise"   → Give me an error 

errors="coerce"  → Convert invalid values to NaN 

errors="ignore"  → Leave invalid values as they are

In [24]:
for col in date_columns:
    orders[col]=pd.to_datetime(
        orders[col],
        errors="coerce"#If Pandas cannot convert a value properly, instead of giving an error, convert that value to NaN
        )

In [25]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [26]:
orders[date_columns].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


If there was an invalid date such as:"not-a-date".Pandas would turn it into NaT (Not a Time)

In [27]:
orders[date_columns].isna().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

### Do the same for reviews

In [28]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [29]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

In [30]:
for col in review_date_columns:
    reviews[col]=pd.to_datetime(
        reviews[col],
        errors="coerce"
    )

In [31]:
reviews[review_date_columns].head()

,review_creation_date,review_answer_timestamp
0,2018-01-18,2018-01-18 21:46:59
1,2018-03-10,2018-03-11 03:05:13
2,2018-02-17,2018-02-18 14:36:24
3,2017-04-21,2017-04-21 22:02:06
4,2018-03-01,2018-03-02 10:26:53


In [32]:
reviews[review_date_columns].isna().sum()

review_creation_date       0
review_answer_timestamp    0
dtype: int64

### Do the same for order items

In [33]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [34]:
order_items["shipping_limit_date"]=pd.to_datetime(order_items["shipping_limit_date"],errors="coerce")

In [35]:
order_items["shipping_limit_date"].head()

0   2017-09-19 09:45:35
1   2017-05-03 11:05:13
2   2018-01-18 14:48:30
3   2018-08-15 10:10:18
4   2017-02-13 13:57:51
Name: shipping_limit_date, dtype: datetime64[us]

In [36]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [37]:
#Number of days between purchase and delivery.
orders["delivery_days"].head()

0     8.0
1    13.0
2     9.0
3    13.0
4     2.0
Name: delivery_days, dtype: float64

In [38]:
orders["delivery_days"].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [39]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [40]:
#A negative value is not an error.
#That means the order arrived 8 days early.
orders["delivery_delay_days"].head()

0    -8.0
1    -6.0
2   -18.0
3   -13.0
4   -10.0
Name: delivery_delay_days, dtype: float64

In [41]:
#How many delivered orders were late?
(orders["delivery_delay_days"] > 0).sum()

np.int64(6535)

In [42]:
#Delivered earlier than estimated.
(orders["delivery_delay_days"] < 0).sum()

np.int64(88649)

In [43]:
#Delivered exactly on the estimated date.
(orders["delivery_delay_days"] == 0).sum()

np.int64(1292)

In [44]:
orders[
    orders["delivery_days"] > 0
][
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0
...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,delivered,2017-03-09 09:54:05,2017-03-17 15:08:01,8.0
99437,63943bddc261676b46f01ca7ac2f7bd8,delivered,2018-02-06 12:58:58,2018-02-28 17:37:56,22.0
99438,83c1379a015df1e13d02aae0204711ab,delivered,2017-08-27 14:46:43,2017-09-21 11:24:17,24.0
99439,11c177c8e97725db2631073c19f07b62,delivered,2018-01-08 21:28:27,2018-01-25 23:32:54,17.0


In [45]:
#Check whether estimated delivery is before purchase
(
    orders["order_estimated_delivery_date"]
    < orders["order_purchase_timestamp"]
).sum()

np.int64(0)

Calculate the late-delivery rate

In [46]:
delivered_orders = orders[
    orders["order_delivered_customer_date"].notna()
]

In [47]:
late_orders = delivered_orders[
    delivered_orders["delivery_delay_days"] > 0
]

In [48]:
late_delivery_rate = (
    len(late_orders) / len(delivered_orders) * 100
)

late_delivery_rate

6.773705377503212

Check the order status logic

In [49]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [50]:
#For each order status, how many orders have an actual customer delivery date?
orders.groupby("order_status")[
    "order_delivered_customer_date"
    ].apply(lambda x: x.notna().sum())

order_status
approved           0
canceled           6
created            0
delivered      96470
invoiced           0
processing         0
shipped            0
unavailable        0
Name: order_delivered_customer_date, dtype: int64

Check order item prices

In [51]:
order_items[["price","freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [52]:
order_items[["price","freight_value"]].isna().sum()

price            0
freight_value    0
dtype: int64

In [ ]:
#A product with a zero price might be suspicious, so we're checking it.
(order_items["price"]<=0).sum()

np.int64(0)

In [57]:
#A negative freight charge makes no business sense, so we're checking specifically for values below zero.
(order_items["freight_value"]<=0).sum()

np.int64(383)

Investigate zero-price items

In [58]:
order_items[
    order_items["price"] <= 0
][
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
]

,order_id,order_item_id,product_id,seller_id,price,freight_value


In [59]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [60]:
dimension_columns=[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

In [62]:
(products[dimension_columns]<0).sum()

product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64

In [64]:
#Zero dimensions might be data-quality problems, but again, we investigate before deciding.
(products[dimension_columns]==0).sum()

product_weight_g     4
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64

### Validate primary identifiers

 These identifiers are extremely important because they're used to connect tables.

In [65]:
customers["customer_id"].isna().sum()

np.int64(0)

In [66]:
orders["order_id"].isna().sum()

np.int64(0)

In [67]:
products["product_id"].isna().sum()

np.int64(0)

In [68]:
sellers["seller_id"].isna().sum()

np.int64(0)

### Check foreign-key integrity

Does every order_items.product_id actually exist in products.product_id?

In [91]:
# ~ : not 
missing_products=~order_items["product_id"].isin(products["product_id"])

missing_products.sum()

np.int64(0)

In [ ]:
# ~ : not 
missing_sellers=~order_items["seller_id"].isin(sellers["seller_id"])

missing_sellers.sum()

np.int64(0)

In [ ]:
# ~ : not 
missing_orders = ~order_items["order_id"].isin(
    orders["order_id"]
)

missing_orders.sum()

np.int64(0)

## Create a validation summary

In [90]:
validation_summary={
    "duplicate_orders": orders["order_id"].duplicated().sum(),
    "negative_delivery_days": (orders["delivery_days"]<0).sum(),
    "negative_prices":(order_items["price"]<0).sum(),
    "negative_freight":(order_items["freight_value"]<0).sum(),
    "invalid_review_score":((reviews["review_score"]<1)|(reviews["review_score"]>5)).sum(),
    "missing_products_referances":missing_products.sum(),
    "missing_seller_references": missing_sellers.sum(),
    "missing_order_references": missing_orders.sum()
}
validation_summary

{'duplicate_orders': np.int64(0),
 'negative_delivery_days': np.int64(0),
 'negative_prices': np.int64(0),
 'negative_freight': np.int64(0),
 'invalid_review_score': np.int64(0),
 'missing_products_referances': np.int64(0),
 'missing_seller_references': np.int64(0),
 'missing_order_references': np.int64(0)}

## Data Quality Findings

The initial validation checks evaluate:

- Missing values
- Duplicate records
- Invalid numeric values
- Date consistency
- Review score validity
- Foreign-key integrity

Raw source files are preserved unchanged in `data/raw/`.

Transformations are applied only to in-memory DataFrames at this stage.

## Save the processed orders

In [ ]:
# this creates orders_cleaned.csv in processed folder 

# index=False: (Pandas has its own row index) It says, dont write pandas internal index into the csv as unnecessary csv column
orders.to_csv("../data/processed/orders_cleaned.csv",index=False)

In [98]:
order_items.to_csv("../data/processed/order_items_cleaned.csv",index=False)



In [99]:

customers.to_csv("../data/processed/customers_cleaned.csv",index=False)

In [97]:
products.to_csv("../data/processed/products_cleaned.csv",index=False)

In [100]:

sellers.to_csv("../data/processed/sellers_cleaned.csv",index=False)

In [101]:

reviews.to_csv("../data/processed/reviews_cleaned.csv",index=False)

In [102]:
payments.to_csv(
    "../data/processed/payments_cleaned.csv",
    index=False
)

In [103]:
geolocation.to_csv(
    "../data/processed/geolocation_cleaned.csv",
    index=False
)

In [104]:
category_translation.to_csv(
    "../data/processed/category_translation_cleaned.csv",
    index=False
)

## Phase 2 Summary

### Validation

The raw datasets were checked for:

- Duplicate order IDs
- Duplicate rows
- Negative prices
- Negative freight values
- Invalid review scores
- Invalid delivery durations
- Missing product references
- Missing seller references
- Missing order references

### Results

The validation checks identified no duplicate order IDs, negative prices, negative freight values, invalid review scores, negative delivery durations, or broken foreign-key references.

### Transformations

The following transformations were applied:

- Converted order timestamps to datetime
- Converted review timestamps to datetime
- Converted shipping limit timestamps to datetime
- Created `delivery_days`
- Created `delivery_delay_days`

The original raw data remains unchanged in `data/raw/`.

Processed datasets are generated in `data/processed/` and are excluded from Git tracking.